In [1]:
import pickle
import pandas as pd
from pathlib import Path
from cmdstanpy import CmdStanModel

DATA_PATH = Path("../EDA/data/dataset.csv")
STAN_FILE = Path("flat.stan")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

N_CHAINS = 4
N_WARMUP = 1000
N_SAMPLES = 1000
SEED = 42

assert DATA_PATH.exists(), f"dataset.csv not found at {DATA_PATH}"
assert STAN_FILE.exists(), f"Stan file not found at {STAN_FILE}"

def load_and_validate(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = ["choice", "RT", "difficulty_bin"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df = df.dropna(subset=required)
    df = df[(df["RT"] > 0.05)]

    print(f"Loaded: {len(df):,} trials")
    print(f"RT range: {df['RT'].min():.3f}s -- {df['RT'].max():.3f}s")
    print(f"Choice balance: {df['choice'].mean():.1%} offensive")
    return df

def build_stan_data(df: pd.DataFrame) -> dict:
    return {
        "N": len(df),
        "choice": df["choice"].astype(int).tolist(),
        "RT": df["RT"].tolist(),
        "difficulty": df["difficulty_bin"].astype(int).tolist()
    }

df = load_and_validate(DATA_PATH)
stan_data = build_stan_data(df)

model = CmdStanModel(stan_file=str(STAN_FILE))
fit = model.sample(
    data=stan_data,
    chains=N_CHAINS,
    iter_warmup=N_WARMUP,
    iter_sampling=N_SAMPLES,
    seed=SEED,
    show_progress=True
)

summary = fit.summary()
summary.to_csv(RESULTS_DIR / "flat_summary.csv")

with open(RESULTS_DIR / "flat_fit.pkl", "wb") as f:
    pickle.dump(fit, f)

summary.loc[["v_easy", "v_hard", "v_diff", "beta", "p_v_easy_gt_v_hard"]]

c:\Users\esabe\AppData\Local\r-miniconda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: 22,263 trials
RT range: 0.100s -- 2.933s
Choice balance: 60.3% offensive


17:05:58 - cmdstanpy - INFO - compiling stan file C:\Users\esabe\Documents\CSCI-4150\Cognitive-Modeling-Capstone-Project\models\flat.stan to exe file C:\Users\esabe\Documents\CSCI-4150\Cognitive-Modeling-Capstone-Project\models\flat.exe


ValueError: Failed to compile Stan model 'C:\Users\esabe\Documents\CSCI-4150\Cognitive-Modeling-Capstone-Project\models\flat.stan'. Console:
/usr/bin/sh: -c: line 0: syntax error near unexpected token `('
/usr/bin/sh: -c: line 0: `expr Microsoft (R) C/C++ Optimizing Compiler Version 19 Copyright (C) Microsoft Corporation  cl : Command line warning D9002 : ignoring unknown option '-dumpversion' cl : Command line error D8003 : missing source filename \>= 8'
Microsoft (R) C/C++ Optimizing Compiler Version 19.39.33523 for x64
Copyright (C) Microsoft Corporation.  All rights reserved.

cl : Command line warning D9002 : ignoring unknown option '-dM'
cl : Command line warning D9002 : ignoring unknown option '-'
cl : Command line error D8003 : missing source filename
stan/lib/stan_math/make/compiler_flags:145: "Stan cannot detect if your compiler has the C++17 standard. If it does, please set STAN_HAS_CXX17=true in your make/local file. C++17 support is mandatory in the next release of Stan. D